# How to discover and access data with EDEN

## Rotterdam Use Case

<hr>

### Importing required libraries

In [ ]:
import requests
import os
import json
from colorama import Fore,Style
import matplotlib.pyplot as plt
from matplotlib import cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import warnings
from destinepyauth import get_token

warnings.filterwarnings("ignore")

#### **Step 1:** Authentication

The HDA endpoint and the corresponding token to access the catalogue are defined as follows

In [ ]:
hdaednpoint = 'https://broker.eden.destine.eu'

In [ ]:
token = get_token("eden").access_token

#### **Step 2:** Browsing the catalogue

The following code extract the full list of datasets that are available in the catalogue. The list is returned as a json object.

In [ ]:
#--------------extracting the list of datasets----------------
getDatasets = requests.get(hdaednpoint + '/api/v1/datasets').json()
datasetList = getDatasets['features']

#--------------visualization of the list----------------
print(Fore.RED + '\033[1m' + 'List of available datasets:')
print('----------------------------------------------------------------------')
print ('\033[0m')

for i in getDatasets['features']:
         print(Style.BRIGHT + Fore.RED + i['metadata']['title'] + Fore.BLACK + "\033[1m" + " --> datasetId "+ "\033[0m" + "= " + i['dataset_id'])


#### **Step 3.** Querables extraction
For a given dataset, specified by its **datasetId**, the following code shows the list of available querables to be used for the data discovery operation. 

In [ ]:

#--------------extracting the list of quearables----------------
datasetID = 'EO:MEEO:DAT:REANALYSIS_ERA5_SINGLE_LEVELS:COG'

getParam = requests.get(os.path.join(hdaednpoint, "api/v1/dataaccess/queryable/",datasetID), headers={"Authorization": "Bearer %s " % token}).json()


#--------------printing the list of querables----------------
print(Fore.BLUE + '\033[1m' + 'List of available querables:')
print('----------------------------------------------------------------------')
print ('\033[0m')
    
for filter,v in getParam['properties'].items():
    print(Style.BRIGHT + Fore.RED + str(filter),Fore.BLACK + str(v))
    print('')

#### **Step 4**. Basic search operation
A simple product search is implemented, providing the datasetID of the desired dataset and the maximum number of results to display. The output will be a Json object containing the metadata of the latest 10 products available in the catalogue. For each product, a selection of its metadata will be printed in the list

In [ ]:

#--------------defining the parameters for the search operation----------------
maxRecords = '10'
data = {
    "dataset_id": datasetID,
    "itemsPerPage": maxRecords,
    "startIndex": "0",
    "subDatasetId":"2m_temperature"
}

##--------------running the search operation----------------
getProducts = requests.post(os.path.join(hdaednpoint, "api/v1/dataaccess/search"), headers={"Authorization": "Bearer %s " % token}, data=json.dumps(data))

results = getProducts.json()


#--------------printing the results of the search----------------
total = results['properties']['totalResults']

if total>int(maxRecords):
    print(Fore.BLACK + 'List of the ' + str(maxRecords) + ' most recent products')
else:
    print(Fore.BLACK + 'List of the ' + str(total)+ ' most recent products')
count = 1
list_result = []
for i in results["features"]:
    print("\033[1;30;1m" + '-----------------------------------------------')
    print("\033[1;30;1m" + "#" + str(count))
    print ('\033[0m')
    print(Style.BRIGHT + Fore.RED + 'product: ' + Fore.GREEN + i['id'])
    print(Style.BRIGHT + Fore.RED + 'start date: ' + Fore.GREEN + i['properties']['startdate'])
    print(Style.BRIGHT + Fore.RED + 'end date: ' + Fore.GREEN +i['properties']['enddate'])
    print(Style.BRIGHT + Fore.RED + 'location: ' + Fore.BLUE + i['properties']['location'])
    count = count+1
    result = {'product_id': i['id'], 'location':i['properties']['location']}
    list_result.append(result)

#### **Step 5.**  Product Download

From the short list of 10 products retrieved in the previous step, one product is directly downloaded in cache 

In [ ]:
select = input('which of the displayed 10 products do you want to download? (type one number in the range 1-10) : ')
#--------------setting the parameters for the download request----------------


data = {
    "dataset_id": datasetID,
    "product_id": list_result[int(select)-1]['product_id'],
    "location": list_result[int(select)-1]['location']}

#--------------running the download request----------------
pdownload = requests.post(
    os.path.join(hdaednpoint, "api/v1/dataaccess/download"),
    headers={"Authorization": "Bearer %s " % token},
    data=json.dumps(data)
)
print(pdownload.json())

#### **Step 6.** Download verification

The following lines of code verify that the product has been definitely downloaded and is available in cache.

In [ ]:
while True:
    hdownload = requests.head(os.path.join(hdaednpoint, "api/v1/dataaccess/download/", pdownload.json()['download_id']),
                              headers={"Authorization": "Bearer %s " % token})
    if hdownload.status_code == 200:
        break
    else:
        print("wait I'm caching")
print('Done!')

In [ ]:
#--------------downloading the product and saving it in the jupyter environment----------------
hdownload = requests.get(os.path.join(hdaednpoint, "api/v1/dataaccess/download/", pdownload.json()['download_id']),
                              headers={"Authorization": "Bearer %s " % token})

with open("response.tif", "wb") as f:
    f.write(hdownload.content)

#### **Step 7.** Data visualization

The downloaded product will be read with the **xarray** library and then plotted on a map, using  the available python packages
for data visualization

In [ ]:
#--------------reading the product with xarray----------------

dataset = xr.open_dataset('./response.tif' )
dataset

In [ ]:
#--------------setting map and gridlines----------------

fig=plt.figure(figsize=(15,15))
cmap = cm.jet
cmap.set_bad('None')
ax = plt.axes(projection=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels=True, linewidth=0.4)
ax.coastlines(resolution='10m',linewidth=0.4,color = 'grey',zorder=3)
ax.add_feature(cfeature.BORDERS, linestyle=':',linewidth=0.7,edgecolor = 'silver')
plt.title('ERA5 Reanalysis - 2m temperature - 2024-04-10T23:00:00Z',fontsize=17, fontweight = 'bold', pad = 15, color = 'red')
gl.xlabels_top = False
gl.ylabels_right = False

#--------------setting the colorbar----------------
img = ax.pcolormesh(dataset.x, dataset.y, dataset.band_data.values[0,:,:]-273.15, cmap = cmap,zorder = 0,vmax = 40,vmin = -40)
cbar = fig.colorbar(img, ax=ax, orientation='horizontal', fraction=0.08, pad=0.05,shrink = 0.8, extend = 'both')
cbar.ax.set_title('[C°]',fontsize = 12, fontweight = 'bold',pad = 8)
cbar.ax.tick_params(labelsize=14)


#--------------saving the map as an image----------------
savefig = plt.savefig('./EDEN_test.png', bbox_inches='tight')

plt.show()

Zoom over Rotterdam to visualize the dataset.

In [ ]:
import display

# Zoom over Rotterdam and plot with the shared display helper.
# adjust zoom below
xmin, xmax = 2, 7
ymin, ymax = 50, 54

# extract relevant data
rotterdam = dataset.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))
rotterdam_t2m = rotterdam.band_data.values[0, :, :] - 273.15
rotterdam_da = xr.DataArray(
    rotterdam_t2m,
    coords={"y": rotterdam.y, "x": rotterdam.x},
    dims=["y", "x"],
    attrs={"units": "C"},
)

# plot
display.map(
    rotterdam_da,
    extent=[xmin, xmax, ymin, ymax],
    vmin=-40,
    vmax=40,
    title="ERA5 Reanalysis 2m Temperature - Rotterdam",
    cmap="coolwarm",
)